In [ ]:
import os
from pathlib import Path

# Localiza a raiz do projeto de forma robusta (funciona em VS Code, JupyterLab, etc.)
def _find_project_root():
    # VS Code expõe o caminho do notebook nesta variável
    nb_file = globals().get('__vsc_ipynb_file__') or locals().get('__vsc_ipynb_file__')
    if nb_file:
        return Path(nb_file).resolve().parent.parent
    # JupyterLab / linha de comandos
    try:
        import ipynbname
        return ipynbname.path().parent.parent
    except Exception:
        pass
    # Último recurso: working directory atual sobe um nível
    cwd = Path().resolve()
    if cwd.name == 'notebooks':
        return cwd.parent
    return cwd

PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
print(f"✓ Working directory: {PROJECT_ROOT}")


In [17]:
import geopandas as gpd
import pandas as pd
import re
from pathlib import Path

# ─────────────────────────────────────────────
# 1. CONFIGURAÇÃO E PARÂMETROS
# ─────────────────────────────────────────────

GPKG_PATH  = "data/raw/edificios_aveiro.gpkg"
PVGIS_PATH = "data/raw/pvgis_aveiro.csv"
CONSUMO_PATH = "data/raw/serie_consumo_cp7_2024_2025_v2.csv"
LAYER_NAME = "postal_code_buildings_assigned"

# Colunas do GPKG
COL_CP7      = "cp7"
COL_FRAG     = "building_postal_fragment_area_m2"  
COL_AREA_TOT = "building_area_m2"                  
COL_N_CP7    = "n_cp7_on_building"                 

# Parâmetros fotovoltaicos
ETA_PAINEL                 = 0.20   # Eficiência do painel (mono-Si típico)
PR                         = 0.80   # Performance Ratio (temperatura, cabos, inversor)
FATOR_OCUPACAO_RESIDENCIAL = 0.40   # Fração do telhado tecnicamente utilizável

In [18]:
def parse_pvgis_horario(filepath: str) -> float:
    p = Path(filepath)
    with open(p, encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    header_idx = None
    for i, line in enumerate(lines):
        if re.match(r"\s*time", line, re.IGNORECASE):
            header_idx = i
            break

    if header_idx is None:
        raise ValueError("Cabeçalho 'time' não encontrado no ficheiro PVGIS.")

    df = pd.read_csv(p, skiprows=header_idx, comment="#", on_bad_lines="skip")
    df.columns = df.columns.str.strip()

    gi_cols = [c for c in df.columns if re.search(r"G\(i\)", c)]
    if not gi_cols:
        raise ValueError("Coluna 'G(i)' não encontrada.")

    df[gi_cols[0]] = pd.to_numeric(df[gi_cols[0]], errors="coerce")
    df = df.dropna(subset=[gi_cols[0]])

    n_horas = len(df)
    n_anos  = round(n_horas / 8760)
    h_total = df[gi_cols[0]].sum() / 1000
    h_anual = h_total / n_anos

    print(f"  [PVGIS] {n_horas:,} horas | {n_anos} ano(s) de dados")
    print(f"  [PVGIS] H(i) total = {h_total:.1f} kWh/m² | H(i) anual = {h_anual:.1f} kWh/m²/ano")
    return h_anual

print("[1/4] A parsear dados horários PVGIS...")
h_anual = parse_pvgis_horario(PVGIS_PATH)

[1/4] A parsear dados horários PVGIS...
  [PVGIS] 52,584 horas | 6 ano(s) de dados
  [PVGIS] H(i) total = 11161.5 kWh/m² | H(i) anual = 1860.2 kWh/m²/ano


In [19]:
print("[2/4] A carregar edifícios de Aveiro...")
gdf = gpd.read_file(GPKG_PATH, layer=LAYER_NAME)
print(f"  [GPKG] {len(gdf):,} registos carregados.")

# Executar diagnóstico corrigido (evitando o bug de alinhamento por índices posicionais)
n_split = (gdf[COL_N_CP7] > 1).sum()
frag_sum = gdf.groupby("polygon_id")[COL_FRAG].sum()
area_tot = gdf.drop_duplicates("polygon_id").set_index("polygon_id")[COL_AREA_TOT]
ratio = (frag_sum / area_tot).dropna()

print(f"  [diag] Edifícios partilhados por 2+ CP7 : {n_split:,}")
print(f"  [diag] Àrea total mapeada (building_area_m2)  : {gdf.drop_duplicates('polygon_id')[COL_AREA_TOT].sum()/1e6:.2f} km²")
print(f"  [diag] Àrea útil de fragmentos (soma total)   : {gdf[COL_FRAG].sum()/1e6:.2f} km²")
print(f"  [diag] Edifícios com rácio fragmento/total correto (==1.0): {(ratio.round(4) == 1.0).sum()} de {len(area_tot)}")

if (ratio.round(4) != 1.0).any():
    print(f"  [AVISO] {(ratio.round(4) != 1.0).sum()} edifícios com discrepâncias geométricas residuais.")

[2/4] A carregar edifícios de Aveiro...
  [GPKG] 17,559 registos carregados.
  [diag] Edifícios partilhados por 2+ CP7 : 2,120
  [diag] Àrea total mapeada (building_area_m2)  : 2.45 km²
  [diag] Àrea útil de fragmentos (soma total)   : 2.45 km²
  [diag] Edifícios com rácio fragmento/total correto (==1.0): 16019 de 16019


In [21]:
print("[3/4] A calcular produção fotovoltaica potencial por fragmento...")
gdf["area_util_m2"]     = gdf[COL_FRAG] * FATOR_OCUPACAO_RESIDENCIAL
gdf["potencia_kwp"]     = gdf["area_util_m2"] * ETA_PAINEL
gdf["producao_kwh_ano"] = h_anual * gdf["area_util_m2"] * ETA_PAINEL * PR

print(f"  Produção total municipal estimada : {gdf['producao_kwh_ano'].sum()/1e6:.2f} GWh/ano")
print(f"  Potência total instalável nominal : {gdf['potencia_kwp'].sum()/1000:.1f} MWp")

# Agregação por código postal
df_producao_cp7 = (
    gdf.groupby(COL_CP7, dropna=False)
    .agg(
        n_edificios              = ("polygon_id", "count"),
        fragment_area_total_m2   = (COL_FRAG, "sum"),
        area_util_total_m2       = ("area_util_m2", "sum"),
        potencia_total_kwp       = ("potencia_kwp", "sum"),
        producao_total_kwh       = ("producao_kwh_ano", "sum"),
        n_edificios_partilhados  = (COL_N_CP7, lambda x: (x > 1).sum()),
    )
    .reset_index()
)
df_producao_cp7.to_csv("data/processed/producao_pv_cp7.csv", index=False, encoding= "utf-8-sig")
print("  ✓ Ficheiro 'producao_pv_cp7.csv' guardado com sucesso.")

[3/4] A calcular produção fotovoltaica potencial por fragmento...
  Produção total municipal estimada : 292.08 GWh/ano
  Potência total instalável nominal : 196.3 MWp
  ✓ Ficheiro 'producao_pv_cp7.csv' guardado com sucesso.


In [23]:
print("[4/4] A carregar e tratar série temporal de consumo...")
df_cons = pd.read_csv(CONSUMO_PATH, sep=",", encoding="utf-8-sig")
df_cons.columns = df_cons.columns.str.strip()

# Filtro de integridade: Mínimo de 70 horas de leitura registadas
LIMIAR_REGISTOS = 70
agg_cons = df_cons.groupby("cp7")["energia_ativa_kwh"].agg(['sum', 'count']).reset_index()
agg_cons = agg_cons.rename(columns={"sum": "consumo_acumulado_kwh", "count": "n_registos"})

consumo_valido = agg_cons[agg_cons["n_registos"] >= LIMIAR_REGISTOS].copy()
print(f"  CP7s originais no consumo: {len(agg_cons)} | Removidos por falta de dados (<{LIMIAR_REGISTOS}h): {len(agg_cons) - len(consumo_valido)}")

# Extrapolação Anual Estatisticamente Robusta
consumo_valido["consumo_anual_kwh"] = (consumo_valido["consumo_acumulado_kwh"] / consumo_valido["n_registos"]) * 8760
print(f"  Consumo total extrapolado do município (antes do cruzamento): {consumo_valido['consumo_anual_kwh'].sum() / 1e6:.2f} GWh/ano")

[4/4] A carregar e tratar série temporal de consumo...
  CP7s originais no consumo: 721 | Removidos por falta de dados (<70h): 0
  Consumo total extrapolado do município (antes do cruzamento): 502.43 GWh/ano


In [24]:
print("A cruzar matrizes de Produção Potencial e Consumo Real...")

# Análise de cobertura espacial
cp7_gpkg = set(df_producao_cp7["cp7"].unique())
cp7_cons = set(consumo_valido["cp7"].unique())
print(f"  CP7s na malha de edifícios (GPKG): {len(cp7_gpkg)}")
print(f"  CP7s com leituras válidas de consumo : {len(cp7_cons)}")
print(f"  Interseção perfeita (Match)          : {len(cp7_gpkg & cp7_cons)}")
print(f"  Limitação de cobertura (Só no GPKG) : {len(cp7_gpkg - cp7_cons)} CP7s (Sem dados de consumo)")

# Executar o Inner Merge definitivo para a Matriz de Implementação
comparacao_final = pd.merge(
    consumo_valido[["cp7", "consumo_anual_kwh", "n_registos"]],
    df_producao_cp7[["cp7", "producao_total_kwh", "n_edificios", "potencia_total_kwp"]],
    on="cp7",
    how="inner"
)

comparacao_final["balanco_kwh"] = comparacao_final["producao_total_kwh"] - comparacao_final["consumo_anual_kwh"]
comparacao_final["taxa_autossuficiencia_%"] = (comparacao_final["producao_total_kwh"] / comparacao_final["consumo_anual_kwh"]) * 100

total_consumo_m = comparacao_final["consumo_anual_kwh"].sum() / 1e6
total_producao_m = comparacao_final["producao_total_kwh"].sum() / 1e6
taxa_global = (total_producao_m / total_consumo_m) * 100

print("\n=======================================================")
print("                  MÉTRICAS FINAIS")
print("=======================================================")
print(f" Consumo Total Validado (Zona do Merge) : {total_consumo_m:.2f} GWh/ano")
print(f" Produção Solar Potencial (Zona do Merge): {total_producao_m:.2f} GWh/ano")
print(f" Taxa de Autossuficiência Global         : {taxa_global:.1f}%")
print("=======================================================")

# Guardar dataset final limpo para o próximo notebook
comparacao_final.to_csv("data/processed/comparacao_final_limpa_cp7.csv", index=False)
print("\n✓ Ficheiro final 'comparacao_final_limpa_cp7.csv' gerado para o QGIS/Matriz.")

A cruzar matrizes de Produção Potencial e Consumo Real...
  CP7s na malha de edifícios (GPKG): 936
  CP7s com leituras válidas de consumo : 721
  Interseção perfeita (Match)          : 558
  Limitação de cobertura (Só no GPKG) : 378 CP7s (Sem dados de consumo)

                  MÉTRICAS FINAIS
 Consumo Total Validado (Zona do Merge) : 427.10 GWh/ano
 Produção Solar Potencial (Zona do Merge): 225.43 GWh/ano
 Taxa de Autossuficiência Global         : 52.8%

✓ Ficheiro final 'comparacao_final_limpa_cp7.csv' gerado para o QGIS/Matriz.
